# Custom ReAct Agent using LangGraph StateGraph

This notebook builds a custom, production-grade ReAct (Reason + Act) agent loop from scratch using LangGraph `StateGraph`.

### Architecture Flow:
1. **`START` -> `agent`**: The LLM inspects the conversation messages and decides whether to respond directly or invoke tools.
2. **`should_continue` conditional edge**: Checks if the model generated any `tool_calls`.
   - If tools were called -> route to **`tools`** node.
   - If no tools were called -> route to **`END`**.
3. **`tools` -> `agent`**: Executes the requested tools via `ToolNode` and appends `ToolMessage`s back into the message state, looping back to the agent for further reasoning.

In [ ]:
from dotenv import load_dotenv
from pathlib import Path
import os

# Locate and load the .env file containing the OPENAI_API_KEY
env_path = Path.cwd().parent.parent / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent.parent.parent / ".env"
load_dotenv(env_path)

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in environment or .env file.")


In [ ]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langgraph.graph import MessagesState, START, StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from rich import print


In [ ]:
# Initialize the ChatOpenAI model
model = ChatOpenAI(
    model="gpt-4.1-mini",
    api_key=api_key,
    temperature=0,
)


In [ ]:
# Define custom tools with type annotations and clear docstrings

@tool
def add(a: float, b: float) -> float:
    """Add two numbers together (a + b)."""
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together (a * b)."""
    return a * b

@tool
def get_stock_price(ticker: str) -> str:
    """Fetch the latest mock stock price for a given company ticker symbol (e.g. AAPL, NVDA, GOOGL)."""
    mock_prices = {
        "AAPL": "$230.50",
        "GOOGL": "$175.20",
        "MSFT": "$420.10",
        "NVDA": "$135.80",
    }
    return mock_prices.get(ticker.upper(), f"Stock price for '{ticker}' not found.")

tools = [add, multiply, get_stock_price]


In [ ]:
# Conditional Edge Routing Function

def should_continue(state: MessagesState) -> str:
    """Inspects the latest message to determine if tools should be invoked or the loop should terminate."""
    messages = state["messages"]
    last_message = messages[-1]
    
    # If the model requested tool calls, route to the 'tools' node
    if getattr(last_message, "tool_calls", None):
        return "tools"
    
    # If no tool calls were generated, terminate the execution loop
    return END


In [ ]:
# Custom Agent Factory Function

def create_custom_agent(
    model: ChatOpenAI,
    tools: list,
    system_prompt: str | SystemMessage | None = None,
    checkpointer = None,
    parallel_tool_calls: bool | None = None,
):
    """Factory function to build and compile a custom ReAct style agent loop.
    
    Args:
        model: The base ChatOpenAI instance.
        tools: List of @tool functions available to the agent.
        system_prompt: Optional system instructions guiding the agent's behavior.
        checkpointer: Optional persistence checkpointer (e.g. InMemorySaver) for multi-turn memory.
        parallel_tool_calls: Whether tools can be called in parallel or strictly sequentially.
    """
    # Configure tool binding options
    bind_kwargs = {}
    if parallel_tool_calls is not None:
        bind_kwargs["parallel_tool_calls"] = parallel_tool_calls
        
    model_with_tools = model.bind_tools(tools, **bind_kwargs) if tools else model
    tool_node = ToolNode(tools) if tools else None
    
    # Model execution node
    def call_model(state: MessagesState):
        messages = state["messages"]
        # Prepend system prompt only if specified and not already present at start
        if system_prompt:
            if not messages or not isinstance(messages[0], SystemMessage):
                sys_msg = (
                    system_prompt
                    if isinstance(system_prompt, SystemMessage)
                    else SystemMessage(content=system_prompt)
                )
                messages = [sys_msg] + messages
                
        response = model_with_tools.invoke(messages)
        return {"messages": [response]}
    
    # Construct the graph workflow
    workflow = StateGraph(MessagesState)
    
    # Add the primary agent reasoning node
    workflow.add_node("agent", call_model)
    workflow.add_edge(START, "agent")
    
    # Wire conditional edges and loop if tools are present
    if tools:
        workflow.add_node("tools", tool_node)
        workflow.add_conditional_edges(
            "agent",
            should_continue,
            {"tools": "tools", END: END}
        )
        workflow.add_edge("tools", "agent")  # Loop back to agent after tools execute
    else:
        workflow.add_edge("agent", END)
    
    return workflow.compile(checkpointer=checkpointer)


In [ ]:
# Compile the custom agent with memory support
checkpointer = InMemorySaver()

custom_agent = create_custom_agent(
    model=model,
    tools=tools,
    system_prompt="""You are a versatile, precise assistant equipped with specialized tools.
Use arithmetic tools for calculations and the stock price tool for financial lookup.
Always explain your reasoning clearly and concisely.""",
    checkpointer=checkpointer,
    parallel_tool_calls=False,
)

print("[bold green]Custom Agent compiled successfully![/bold green]")


In [ ]:
# Test 1: Single-turn execution with real-time step streaming
config = {"configurable": {"thread_id": "conversation-1"}}
query = "What is the stock price of NVDA? If I buy 5 shares, how much will it cost in total?"

print(f"[bold cyan]User Query:[/bold cyan] {query}\n")

for chunk in custom_agent.stream(
    {"messages": [HumanMessage(content=query)]},
    config=config,
    stream_mode="updates",
):
    if "agent" in chunk:
        msg = chunk["agent"]["messages"][-1]
        if msg.content:
            print(f"[bold yellow]Agent Thought:[/bold yellow] {msg.content}")
        if getattr(msg, "tool_calls", None):
            for tc in msg.tool_calls:
                print(f"[bold blue]Tool Call:[/bold blue] {tc['name']} with args {tc['args']}")
                
    if "tools" in chunk:
        for t_msg in chunk["tools"]["messages"]:
            print(f"[bold magenta]Tool Output:[/bold magenta] {t_msg.content}\n")


In [ ]:
# Test 2: Multi-turn conversation (Memory Persistence via Checkpointer)
follow_up = "What if I buy 10 shares instead?"
print(f"[bold cyan]Follow-up Query:[/bold cyan] {follow_up}\n")

result = custom_agent.invoke(
    {"messages": [HumanMessage(content=follow_up)]},
    config=config  # Reusing conversation-1 thread_id preserves memory
)

print(f"[bold green]Final Agent Response:[/bold green]\n{result['messages'][-1].content}")
